#  📃 Leaked — India's Exam Paper Crisis, 2004–2026
### An Exploratory Data Analysis of 110 Documented Exam Paper Leak Incidents

---

**Table of Contents**
1. [Imports](#1) &nbsp;·&nbsp; 2. [Load & First Look](#2) &nbsp;·&nbsp; 3. [Data Cleaning](#3) &nbsp;·&nbsp;
4. [Feature Engineering](#4) &nbsp;·&nbsp; 5. [Data Dictionary & Coverage](#5) &nbsp;·&nbsp;
6. [Exploratory Data Analysis](#6) &nbsp;·&nbsp; 7. [Limitations](#7) &nbsp;·&nbsp; 8. [Conclusion](#8)

> **A note on this revision:** this notebook fixes a date-parsing bug, a plotting parameter bug, and a
> misleading-average issue found in the original draft; tightens the exam-sector categorization (the
> catch-all "Other" bucket shrinks from 25.5% to 5.5% of incidents); adds two analyses the cleaning steps
> had prepared for but never used (conducting-body repeat offenders, source confidence / leak-status
> verification); and adds explicit reporting-coverage caveats throughout so every average is honest about
> its denominator. Every output below was generated by a single, linear, top-to-bottom run of the cells
> as they appear — nothing here is a leftover from an earlier, different run.

## 1. Import Libraries <a id='1'></a>
NumPy, pandas, Matplotlib and Seaborn cover everything needed here: data wrangling and the full chart set. A single theme is configured once so every chart below shares the same visual language instead of each cell picking its own palette.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ---- One consistent visual theme for the whole notebook ----
sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": "#3A4A5C",
    "axes.labelcolor": "#2A2F36",
    "text.color": "#2A2F36",
    "figure.dpi": 110,
})

# A small, deliberate palette used consistently across every chart below,
# rather than a different named seaborn palette per cell.
C = {
    "primary":  "#A6423D",   # deep red  -- main emphasis / "State" / leaks
    "secondary":"#3A4A5C",   # slate navy -- contrast series / "Central"
    "accent":   "#C9973F",   # muted gold -- tertiary accent
    "muted":    "#94A0AA",   # neutral grey -- de-emphasis / "Other"
    "good":     "#4C6B4F",   # muted green -- "Confirmed" / verified only
}
def seq_palette(n, color=C["primary"]):
    """Sequential ramp with exactly n colors -- avoids seaborn's palette-length mismatch warnings."""
    return sns.light_palette(color, n_colors=n, reverse=True)

def annotate_bars(ax, fmt="{:.0f}", orient="v", **kwargs):
    """Label each bar with its data value. orient='v': vertical bars, value = height.
    orient='h': horizontal bars, value = width. (Bar *width* on a vertical chart is just
    the fixed geometric bar thickness, e.g. 0.8 -- never the data value; this must be
    explicit rather than guessed, which was the source of an earlier bug.)"""
    for container in ax.containers:
        if orient == "v":
            labels = [fmt.format(v.get_height()) if v.get_height() else "" for v in container]
        else:
            labels = [fmt.format(v.get_width()) if v.get_width() else "" for v in container]
        ax.bar_label(container, labels=labels, padding=3, fontweight="bold", fontsize=10, **kwargs)

## 2. Load the Dataset & First Look <a id='2'></a>
The CSV is expected in the same folder as this notebook (swap the path below if yours lives elsewhere).

In [2]:
df = pd.read_csv('paper_leaks.csv')
df.shape

(110, 18)

In [3]:
df.head()

,incident_id,date,era,exam_name,conducting_body,body_type,area,leak_status,action_taken,note,arrests,convictions,aspirants_affected,linked_deaths,deaths_note,source_name,source_url,confidence
0,PL-0001,2004-04-11,UPA (2004-May2014),All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004,Central Board of Secondary Education (CBSE),Central,All India,Confirmed,Exam cancelled + Arrests-FIR + Probe (CBI/SIT),Physics and chemistry papers leaked in New Del...,2.0,NaN,NaN,NaN,NaN,The Tribune,https://www.tribuneindia.com/2004/20040412/mai...,High
1,PL-0002,2006-06-01,UPA (2004-May2014),Himachal Pradesh Combined Pre-Medical Test (HP...,Himachal Pradesh University,State,Himachal Pradesh,Confirmed,Retest + Arrests-FIR,HP-CPMT medical entrance paper leaked via a to...,NaN,NaN,NaN,NaN,NaN,The Tribune,https://www.tribuneindia.com/2006/20060831/him...,Medium
2,PL-0003,2008-01-01,UPA (2004-May2014),Vyapam Patwari Recruitment Examination 2008,Madhya Pradesh Professional Examination Board ...,State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Accused produced forged educational certificat...,NaN,10.0,NaN,NaN,NaN,The420.in,https://the420.in/vyapam-scam-cbi-court-patwar...,High
3,PL-0004,2009-07-08,UPA (2004-May2014),MP Pre-Medical Test (PMT) 2009,Vyapam (MPPEB),State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Paid solvers/impersonators sat the exam in pla...,NaN,NaN,114.0,NaN,NaN,The Free Press Journal,https://www.freepressjournal.in/education/indo...,High
4,PL-0005,2010-06-06,UPA (2004-May2014),RRB Mumbai 2010 recruitment exam (Assistant Lo...,"Railway Recruitment Board (RRB), Mumbai",Central,All India,Confirmed,Arrests-FIR + Probe (CBI/SIT),CBI registered a case on 15 June 2010 over the...,15.0,10.0,NaN,NaN,NaN,The Week,https://www.theweek.in/wire-updates/national/2...,High


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   incident_id         110 non-null    str    
 1   date                110 non-null    str    
 2   era                 110 non-null    str    
 3   exam_name           110 non-null    str    
 4   conducting_body     110 non-null    str    
 5   body_type           110 non-null    str    
 6   area                110 non-null    str    
 7   leak_status         110 non-null    str    
 8   action_taken        110 non-null    str    
 9   note                110 non-null    str    
 10  arrests             63 non-null     float64
 11  convictions         10 non-null     float64
 12  aspirants_affected  44 non-null     float64
 13  linked_deaths       4 non-null      float64
 14  deaths_note         4 non-null      str    
 15  source_name         110 non-null    str    
 16  source_url         

In [5]:
df.describe().round(2)

,arrests,convictions,aspirants_affected,linked_deaths
count,63.00,10.00,44.00,4.00
mean,26.86,11.40,511626.55,10.25
std,46.70,11.95,890520.28,9.91
min,0.00,0.00,114.00,1.00
25%,4.50,3.25,63250.00,3.25
50%,12.00,10.00,133103.00,8.50
75%,23.50,11.00,600000.00,15.50
max,266.00,34.00,4800000.00,23.00


**First-look notes:**
- 110 rows, 18 raw columns — one row per incident.
- `arrests`, `convictions`, `aspirants_affected`, `linked_deaths` are heavily sparse (counts of 63, 10, 44 and 4
  respectively out of 110 — visible in `.describe()` above, which only counts non-null values). This sparsity
  is real and important: it means most of this notebook's incident-level averages describe *"where reported,"*
  not *"in general."* We'll carry that distinction through every relevant section rather than only mentioning
  it once here.

## 3. Cleaning the Dataset <a id='3'></a>

### 3.1 Standardizing the Date Column
The raw `date` column is ISO-formatted text (`YYYY-MM-DD`), so we parse it without forcing a mismatched explicit format -- forcing the wrong `format=` string here is a classic silent-failure trap (it can produce `NaT` for every row, or worse, silently swap day/month for dates where both happen to be ≤12).

In [6]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print('Unparseable dates:', df['date'].isna().sum(), 'out of', len(df))
df['year'] = df['date'].dt.year
df[['incident_id','date','year']].head()

Unparseable dates: 0 out of 110


,incident_id,date,year
0,PL-0001,2004-04-11,2004
1,PL-0002,2006-06-01,2006
2,PL-0003,2008-01-01,2008
3,PL-0004,2009-07-08,2009
4,PL-0005,2010-06-06,2010


### 3.2 Handling Missing Numeric Values
`arrests`, `convictions`, `aspirants_affected` and `linked_deaths` are frequently unreported rather than genuinely zero. We coerce them to numeric but deliberately **do not** fill missing values with 0: doing so before averaging would silently treat *"not reported"* as *"confirmed none"* and understate every mean. We keep `NaN` as `NaN` -- `.sum()` already ignores it correctly -- and add explicit `_reported` flags so any later reader can see exactly which incidents are contributing to a given average.

In [7]:
numeric_cols = ['arrests', 'convictions', 'aspirants_affected', 'linked_deaths']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[f'{col}_reported'] = df[col].notna()

coverage = pd.DataFrame({
    'reported_n': [df[f'{c}_reported'].sum() for c in numeric_cols],
    'reported_pct': [round(df[f'{c}_reported'].mean()*100,1) for c in numeric_cols],
}, index=numeric_cols)
coverage

,reported_n,reported_pct
arrests,63,57.3
convictions,10,9.1
aspirants_affected,44,40.0
linked_deaths,4,3.6


### 3.3 Stripping Whitespace & Standardizing Case
Several text columns mix casing and stray whitespace from manual data entry.

In [8]:
for col in ['conducting_body', 'body_type', 'leak_status']:
    df[col] = df[col].str.strip().str.title()
df['leak_status'].value_counts()

leak_status
Confirmed    89
Alleged      13
Denied        5
Suspected     3
Name: count, dtype: int64

### 3.4 Correcting `leak_status` & `conducting_body` Labels
Manual entry produced spelling/casing variants of the same real-world entity (e.g. several ways of writing "Vyapam" or "CBSE"). We map these to single canonical labels so later `groupby` counts aren't artificially split.

In [9]:
corrections = {
    # leak_status variants
    'Suspected': 'Alleged',
    # conducting_body variants -> canonical short names
    # (verified against the full raw unique-value list -- note Chhattisgarh's
    # "CG Vyapam/CGPEB" is a distinct state body and is deliberately NOT merged
    # into Madhya Pradesh's Vyapam despite the shared nickname)
    'Cbse': 'CBSE',
    'Central Board Of Secondary Education (Cbse)': 'CBSE',
    'Ssc': 'SSC',
    'Staff Selection Commission (Ssc)': 'SSC',
    'All India Institute Of Medical Sciences (Aiims)': 'AIIMS',
    'Aiims New Delhi': 'AIIMS',
    'Vyapam (Mppeb)': 'Vyapam (MPPEB)',
    'Madhya Pradesh Professional Examination Board (Vyapam)': 'Vyapam (MPPEB)',
    'Madhya Pradesh Professional Examination Board (Vyapam/Mpesb)': 'Vyapam (MPPEB)',
    'Madhya Pradesh Professional Examination Board (Vyapam/Mppeb)': 'Vyapam (MPPEB)',
    'Madhya Pradesh Employees Selection Board (Esb)': 'MP Employees Selection Board (ESB)',
    'Madhya Pradesh Employees Selection Board (Esb, Formerly Peb)': 'MP Employees Selection Board (ESB)',
}
df['leak_status'] = df['leak_status'].replace(corrections)
df['conducting_body'] = df['conducting_body'].replace(corrections)
print('leak_status categories after cleaning:', df['leak_status'].unique().tolist())
df['leak_status'].value_counts()

leak_status categories after cleaning: ['Confirmed', 'Alleged', 'Denied']


leak_status
Confirmed    89
Alleged      16
Denied        5
Name: count, dtype: int64

### 3.5 Cleaning the `action_taken` Column
This column mixes spacing around `+`, inconsistent ordering of the same actions, and varying probe-agency
abbreviations. We standardize spacing and ordering so identical action sets compare equal, then derive a
high-level `action_summary` for charting.

In [10]:
df['action_taken'] = df['action_taken'].str.replace(r'\s*\+\s*', ' + ', regex=True)

def sort_actions(action_string):
    if pd.isna(action_string):
        return action_string
    actions = [a.strip() for a in action_string.split('+')]
    actions.sort()
    return ' + '.join(actions)

df['action_taken'] = df['action_taken'].apply(sort_actions)

def summarize_action(action):
    a = str(action).lower()
    if 'cancelled' in a or 'retest' in a:
        return 'Exam Cancelled / Retest'
    elif 'arrest' in a and 'probe' in a:
        return 'Arrests & Investigation (No Cancellation)'
    elif 'arrest' in a:
        return 'Arrests Only (No Cancellation)'
    elif 'probe' in a:
        return 'Investigation Ongoing (No Arrests Yet)'
    else:
        return 'No Action Reported'

df['action_summary'] = df['action_taken'].apply(summarize_action)
df[['action_taken', 'action_summary']].head(6)

,action_taken,action_summary
0,Arrests-FIR + Exam cancelled + Probe (CBI/SIT),Exam Cancelled / Retest
1,Arrests-FIR + Retest,Exam Cancelled / Retest
2,Arrests-FIR + Probe (CBI/SIT),Arrests & Investigation (No Cancellation)
3,Arrests-FIR + Probe (CBI/SIT),Arrests & Investigation (No Cancellation)
4,Arrests-FIR + Probe (CBI/SIT),Arrests & Investigation (No Cancellation)
5,Arrests-FIR + Probe (CBI/SIT),Arrests & Investigation (No Cancellation)


### 3.6 Normalizing the `area` Column into a Clean `state` Field
Locations mix state names with district/city detail in parentheses (and occasionally a second location after a `/`), which would fragment a geographic rollup. We keep the raw text in `area` and add a clean `state` column for aggregation.

In [11]:
import re

def normalize_state(area):
    if area == 'All India':
        return 'All India'
    primary = area.split(' / ')[0].strip()
    primary = re.sub(r'\s*\([^)]*\)\s*$', '', primary).strip()
    return primary

df['state'] = df['area'].apply(normalize_state)
df[['area','state']].drop_duplicates().head(8)

,area,state
0,All India,All India
1,Himachal Pradesh,Himachal Pradesh
2,Madhya Pradesh,Madhya Pradesh
6,Andaman & Nicobar Islands,Andaman & Nicobar Islands
7,Uttar Pradesh (Kanpur/Lucknow),Uttar Pradesh
8,Chhattisgarh,Chhattisgarh
9,Uttar Pradesh,Uttar Pradesh
13,Delhi,Delhi


## 4. Feature Engineering <a id='4'></a>

### 4.1 Categorizing Exams by Sector
The original keyword rules left a 25.5%-of-all-incidents "Other" bucket -- by far the *largest* single
category, which made the sector breakdown less informative than it should be. Looking at what actually fell
into "Other" (SSC/PCS-style admin exams, revenue-officer roles like Patwari/Tehsildar/Lekhpal, railway
recruitment, and a few teaching-adjacent titles like "Assistant Professor") let us extend the keyword rules
so the catch-all shrinks to the incidents that are genuinely one-offs.

In [12]:
def categorize_exam(name):
    n = str(name).lower()
    if any(w in n for w in ['medical','neet','aipmt','pmt','health','nurse','aiims','pharmacist']):
        return 'Medical / Healthcare'
    elif any(w in n for w in ['police','constable','sub-inspector','army','guard']):
        return 'Police / Defence'
    elif any(w in n for w in ['railway','rrb','loco pilot','rrc ']):
        return 'Railways / Transport'
    elif any(w in n for w in ['teacher','tet','school','education','professor','ugc-net','ugc net']):
        return 'Teaching / Education'
    elif any(w in n for w in ['engineer','aieee','technical']):
        return 'Engineering / Technical'
    elif any(w in n for w in ['board','class 10','class 12','hsc','sslc']):
        return 'School Board Exams'
    elif any(w in n for w in ['psc','pcs','civil service','administrative','clerk','patwari','ssc','cgl',
                                'tehsildar','lekhpal','talathi','panchayat','judicial','hcs ','office assistant']):
        return 'Civil Services / Admin'
    else:
        return 'Other State/Central Exams'

df['exam_category'] = df['exam_name'].apply(categorize_exam)
cat_check = df['exam_category'].value_counts()
print('"Other" bucket: {} incidents ({:.1f}% of 110) -- down from 28 incidents (25.5%) in the original rules'
      .format(cat_check.get('Other State/Central Exams', 0), cat_check.get('Other State/Central Exams', 0)/110*100))
cat_check

"Other" bucket: 6 incidents (5.5% of 110) -- down from 28 incidents (25.5%) in the original rules


exam_category
Civil Services / Admin       32
Medical / Healthcare         20
Police / Defence             18
Teaching / Education         15
School Board Exams            9
Engineering / Technical       7
Other State/Central Exams     6
Railways / Transport          3
Name: count, dtype: int64

### 4.2 Standardizing Equivalent Exam Names
The same real-world exam is often entered under several names across years (NEET / AIPMT / state PMTs are historically the same medical-entrance lineage, for example). We unify these so repeat-target analysis isn't fragmented by year-specific naming.

In [13]:
def standardize_exam_name(exam):
    e = str(exam).lower()
    if any(k in e for k in ['neet', 'aipmt', 'pmt', 'pre-medical']):
        return 'NEET / Pre-Medical Entrance (UG)'
    elif 'cbse' in e and ('class' in e or 'board' in e):
        return 'CBSE Class 10/12 Board Exams'
    elif any(k in e for k in ['tre', 'tet', 'teacher eligibility']):
        return 'State Teacher Recruitment/Eligibility (TET/TRE)'
    elif 'patwari' in e:
        return 'Patwari Recruitment Exam'
    elif 'constable' in e and 'police' in e:
        return 'Police Constable Recruitment'
    elif 'sub-inspector' in e or ' si ' in e:
        return 'Police Sub-Inspector (SI) Recruitment'
    elif 'combined competitive' in e or 'psc' in e:
        return 'State PSC / Combined Competitive Exam'
    elif 'engineer' in e and any(k in e for k in ['junior', 'assistant', 'je', 'ae']):
        return 'Junior/Assistant Engineer (JE/AE) Recruitment'
    else:
        return exam

df['standardized_exam_name'] = df['exam_name'].apply(standardize_exam_name)
df['standardized_exam_name'].value_counts().head(6)

standardized_exam_name
NEET / Pre-Medical Entrance (UG)                   13
State Teacher Recruitment/Eligibility (TET/TRE)    10
State PSC / Combined Competitive Exam               8
Police Constable Recruitment                        7
Junior/Assistant Engineer (JE/AE) Recruitment       6
Patwari Recruitment Exam                            4
Name: count, dtype: int64

### 4.3 Preparing Conducting-Body Analysis (Repeat Offenders)
The cleaning step in 3.4 already canonicalized `conducting_body` labels (merging Vyapam/CBSE/ESB spelling variants) -- here we simply confirm it's ready to rank directly, which we'll use in section 6.2.

In [14]:
df['conducting_body'].value_counts().head(10)

conducting_body
Vyapam (MPPEB)                                                   10
CBSE                                                              9
Uttarakhand Subordinate Service Selection Commission (Uksssc)     4
Rajasthan Staff Selection Board (Rsmssb)                          4
Nta                                                               4
Rajasthan Public Service Commission (Rpsc)                        3
Bihar Public Service Commission (Bpsc)                            3
Jharkhand Staff Selection Commission (Jssc)                       3
AIIMS                                                             2
SSC                                                               2
Name: count, dtype: int64

In [15]:
df.shape

(110, 27)

## 5. Data Dictionary & Coverage Summary <a id='5'></a>
Before charting anything, it's worth being explicit about what each field means and -- critically -- how completely it's actually reported. This table is the single reference point for "where reported" denominators used throughout section 6.

In [16]:
data_dict = pd.DataFrame([
    ('incident_id',          'Unique ID, PL-0001..PL-0110',                         '100%'),
    ('date / year',          'Incident date (parsed) and its year',                 '100%'),
    ('era',                  'UPA (2004-May 2014) or NDA (May 2014-present)',       '100%'),
    ('exam_name',            'Full exam name as reported',                          '100%'),
    ('exam_category',        'Engineered: 8-sector rollup (section 4.1)',           '100%'),
    ('standardized_exam_name','Engineered: unifies equivalent exam names',          '100%'),
    ('conducting_body',      'Cleaned canonical name of the exam authority',        '100%'),
    ('body_type',            'State or Central authority',                          '100%'),
    ('area / state',         'Raw location text / cleaned state name',              '100%'),
    ('leak_status',          'Confirmed, Alleged, or Denied',                       '100%'),
    ('action_taken',         'Cleaned free text of the official response',          '100%'),
    ('action_summary',       'Engineered: 5-bucket rollup of action_taken',         '100%'),
    ('arrests',              'Count of arrests, where reported',                    '57.3%'),
    ('convictions',          'Count of convictions, where reported',                '9.1%'),
    ('aspirants_affected',   'Candidates affected, where reported',                 '40.0%'),
    ('linked_deaths',        'Press-reported deaths linked to the incident',        '3.6%'),
], columns=['Column', 'Meaning', 'Coverage'])

data_dict.style.set_properties(**{'text-align':'left'}).set_table_styles(
    [{'selector':'th','props':[('text-align','left'), ('background-color', C['secondary']), ('color','white')]}]
).hide(axis='index')

Column,Meaning,Coverage
incident_id,"Unique ID, PL-0001..PL-0110",100%
date / year,Incident date (parsed) and its year,100%
era,UPA (2004-May 2014) or NDA (May 2014-present),100%
exam_name,Full exam name as reported,100%
exam_category,Engineered: 8-sector rollup (section 4.1),100%
standardized_exam_name,Engineered: unifies equivalent exam names,100%
conducting_body,Cleaned canonical name of the exam authority,100%
body_type,State or Central authority,100%
area / state,Raw location text / cleaned state name,100%
leak_status,"Confirmed, Alleged, or Denied",100%


> **Reading this table:** the top block (100%-covered fields) supports incident-level counting -- "how many
> incidents," "which state," "which sector" -- with full confidence. The bottom block (`arrests` down to
> `linked_deaths`) supports **magnitude** questions -- "how many people," "how many arrests" -- but only for
> the fraction of incidents where a number was ever publicly reported. Every average built on those four
> columns in section 6 is explicitly labeled *"where reported."*